In [2]:
pip install spacy 

Note: you may need to restart the kernel to use updated packages.


In [4]:
!python -m spacy download en_core_web_sm

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 978, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno -3] Temporary failure in name resolution

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
             

In [6]:
import spacy

nlp = spacy.load("en_core_web_sm")
print("spaCy ready!")

spaCy ready!


Task 1.1: Extract Dates

In [7]:
import re
from datetime import datetime

def extract_dates(text):
    """
    Extract dates in various formats
    Formats: MM/DD/YYYY, DD-MM-YYYY, Month DD, YYYY, YYYY-MM-DD
    """
    
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',  # MM/DD/YYYY
        r'\d{1,2}-\d{1,2}-\d{4}',  # DD-MM-YYYY
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}',  # Month DD, YYYY
        r'\d{4}-\d{2}-\d{2}'  # ISO format
    ]
    
    dates = []
    
    for pattern in patterns:
        matches = re.findall(pattern, text)
        dates.extend(matches)
    
    return dates


# Test
text = "Invoice date: 03/15/2024. Due: March 30, 2024"
print(extract_dates(text))

['03/15/2024', 'March 30, 2024']


Task 1.2: Extract Currency Amounts

In [8]:
import re

def extract_amounts(text):
    """
    Extract currency amounts
    Handles: $1,250.50, 1250.50, $1250
    """
    
    pattern = r'\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'
    amounts = re.findall(pattern, text)
    
    # Convert to float
    cleaned = []
    for amount in amounts:
        # Remove $ and ,
        clean = amount.replace('$', '').replace(',', '')
        
        # Avoid empty strings
        if clean:
            cleaned.append(float(clean))
    
    return cleaned


# Test
text = "Total: $1,250.50. Tax: $125.05. Subtotal: 1125.45"
print(extract_amounts(text))

[1250.5, 125.05, 112.0, 5.45]


Task 1.3: Extract Invoice/Order Numbers

In [11]:
import re

def extract_invoice_number(text):
    """
    Extract invoice/order numbers
    Patterns: INV-2024-001, #12345, ORDER-ABC123
    """
    
    patterns = [
        r'INV-\d{4}-\d{3}',
        r'#\d{5,}',
        r'ORDER-[A-Z0-9]+',
        r'Invoice (?:Number|#):?\s*([A-Z0-9-]+)'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            # Return captured group if exists, otherwise full match
            return match.group(1) if match.groups() else match.group(0)
    
    return None


# Test
text = "Invoice Number: INV-2024-001"
print(extract_invoice_number(text))

INV-2024-001


PART 2: NAMED ENTITY RECOGNITION 

Task 2.1: Basic NER with spaCy 

In [12]:
import spacy

# Load model
nlp = spacy.load("en_core_web_sm")

# Sample invoice text
text = """
Invoice from Acme Corporation
123 Main Street, New York, NY 10001
Contact: John Smith (john@acme.com)
Date: March 15, 2024
Amount Due: $1,250.50
"""

# Process text
doc = nlp(text)

# Extract entities
print("Found entities:\n")

for ent in doc.ents:
    print(f"{ent.text:25} {ent.label_:15} {spacy.explain(ent.label_)}")

Found entities:

Acme Corporation          ORG             Companies, agencies, institutions, etc.
123                       CARDINAL        Numerals that do not fall under another type
Main Street               FAC             Buildings, airports, highways, bridges, etc.
New York                  GPE             Countries, cities, states
10001                     DATE            Absolute or relative dates or periods
John Smith                PERSON          People, including fictional
March 15, 2024            DATE            Absolute or relative dates or periods
1,250.50                  MONEY           Monetary values, including unit


Task 2.2: Extract Specific Entity Types

In [13]:
import spacy

# Load model (make sure it's already downloaded in Kaggle)
nlp = spacy.load("en_core_web_sm")

def extract_entities(text):
    """
    Extract and organize entities by type
    """
    
    doc = nlp(text)
    
    entities = {
        'persons': [],
        'organizations': [],
        'locations': [],
        'dates': [],
        'money': []
    }
    
    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities['persons'].append(ent.text)
        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text)
        elif ent.label_ in ['GPE', 'LOC']:
            entities['locations'].append(ent.text)
        elif ent.label_ == 'DATE':
            entities['dates'].append(ent.text)
        elif ent.label_ == 'MONEY':
            entities['money'].append(ent.text)
    
    return entities


# Test text
text = """
Invoice from Acme Corporation
123 Main Street, New York, NY 10001
Contact: John Smith (john@acme.com)
Date: March 15, 2024
Amount Due: $1,250.50
"""

# Run extraction
result = extract_entities(text)

# Print results
for entity_type, values in result.items():
    print(f"{entity_type}: {values}")

persons: ['John Smith']
organizations: ['Acme Corporation']
locations: ['New York']
dates: ['10001', 'March 15, 2024']
money: ['1,250.50']


Task 2.3: Visualize Entities with displaCy

In [16]:
html = displacy.render(doc, style="ent", page=True) or ""

PA# RT 3: COMPLETE EXTRACTION PIPELINE****

Task 3.1: Build Invoice Processor 

In [18]:
import json
import re
import spacy
import pytesseract
from PIL import Image

# Load spaCy model
nlp = spacy.load("en_core_web_sm")


# ---------- Helper Functions ----------

def extract_dates(text):
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',
        r'\d{1,2}-\d{1,2}-\d{4}',
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}',
        r'\d{4}-\d{2}-\d{2}'
    ]
    
    dates = []
    for pattern in patterns:
        dates.extend(re.findall(pattern, text))
    
    return list(set(dates))


def extract_amounts(text):
    pattern = r'\$?\d+(?:,\d{3})*(?:\.\d{2})?'
    amounts = re.findall(pattern, text)
    
    cleaned = []
    for amount in amounts:
        clean = amount.replace('$', '').replace(',', '')
        if clean:
            cleaned.append(float(clean))
    
    return cleaned


def extract_invoice_number(text):
    patterns = [
        r'INV-\d{4}-\d{3}',
        r'#\d{5,}',
        r'ORDER-[A-Z0-9]+',
        r'Invoice (?:Number|No|#):?\s*([A-Z0-9-]+)'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1) if match.groups() else match.group(0)
    
    return None


def extract_entities(text):
    doc = nlp(text)
    
    entities = {
        'persons': [],
        'organizations': [],
        'locations': [],
        'dates_ner': [],
        'money_ner': []
    }
    
    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities['persons'].append(ent.text)
        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text)
        elif ent.label_ in ['GPE', 'LOC']:
            entities['locations'].append(ent.text)
        elif ent.label_ == 'DATE':
            entities['dates_ner'].append(ent.text)
        elif ent.label_ == 'MONEY':
            entities['money_ner'].append(ent.text)
    
    return entities


# ---------- Main Pipeline ----------

def process_invoice(image_path):
    """
    Complete pipeline: OCR → Extraction → JSON
    """
    
    # Step 1: OCR
    img = Image.open(image_path)
    text = pytesseract.image_to_string(img)
    
    # Step 2: Regex extraction
    invoice_data = {
        'invoice_number': extract_invoice_number(text),
        'dates': extract_dates(text),
        'amounts': extract_amounts(text)
    }
    
    # Step 3: NER extraction
    entities = extract_entities(text)
    invoice_data.update(entities)
    
    # Step 4: Post-processing
    if invoice_data['amounts']:
        invoice_data['total_amount'] = max(invoice_data['amounts'])
    
    if invoice_data['dates']:
        invoice_data['invoice_date'] = invoice_data['dates'][0]
    
    return invoice_data


# ---------- Test ----------

result = process_invoice("/kaggle/input/datasets/anderianisar/ai-lab7/images/6.JPG")
print(json.dumps(result, indent=2))

{
  "invoice_number": null,
  "dates": [
    "26/01/2015"
  ],
  "amounts": [
    3850.0,
    1.0,
    415000001363.0,
    26.0,
    1.0,
    2015.0,
    16.0,
    13.0,
    1.0,
    0.0,
    2.0,
    74.0,
    0.0,
    1.0,
    16.0,
    0.0,
    1.0,
    13.0,
    0.0,
    1.0,
    72.0,
    0.0,
    175.0,
    0.0,
    175.0,
    0.0,
    200.0,
    0.0,
    2.0,
    2.0,
    96.0,
    81.0,
    9565.0,
    16.0,
    14.0
  ],
  "persons": [
    "Cinst"
  ],
  "organizations": [
    "MOMI & TOY",
    "Cashier\nRept"
  ],
  "locations": [],
  "dates_ner": [
    "3850",
    "9565 16"
  ],
  "money_ner": [],
  "total_amount": 415000001363.0,
  "invoice_date": "26/01/2015"
}


Ta# sk 3.2: Save Results as JSON****

In [21]:
import json

# Save to JSON file
output_file = "extracted_data.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2)

print(f"Results saved to {output_file}")

Results saved to extracted_data.json
